## Env

In [1]:
!pip install unsloth wandb datasets transformers \
    peft accelerate bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
import wandb
wandb.login()  # paste API key from wandb.ai/authorize

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: laylawakeup (laylawakeup-testing) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## data loading

In [4]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 108.1 MB/s eta 0:00:00


In [11]:
from huggingface_hub import list_repo_files

files = list(list_repo_files("theatticusproject/cuad", repo_type="dataset"))
print(files)

['.gitattributes', 'CUAD_v1/CUAD v1 ReadMe _ Datasheet/CUAD_v1_README.txt', 'CUAD_v1/CUAD v1 ReadMe _ Datasheet/Datasheet_for_Contract_Understanding_Atticus_Dataset_(CUAD)_v1.pdf', 'CUAD_v1/CUAD_v1.json', 'CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement.pdf', 'CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf', 'CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.pdf', 'CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.pdf', 'CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/SouthernStarEnergyInc_20051202_SB-2A_EX-9_801890_EX-9_Affiliate Agreement.pdf', 'CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/SteelVaultCorp_20081224_10-K_EX-10.16_3074935_EX-10

In [12]:
from huggingface_hub import hf_hub_download
import json

# Correct path inside the repository
file_path = hf_hub_download(
    repo_id="theatticusproject/cuad",
    filename="CUAD_v1/CUAD_v1.json",
    repo_type="dataset"
)

with open(file_path) as f:
    cuad = json.load(f)

print(f"Number of contracts: {len(cuad['data'])}")
print("First QA example:", cuad['data'][0]['paragraphs'][0]['qas'][0])

CUAD_v1/CUAD_v1.json:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

Number of contracts: 510
First QA example: {'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}], 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'is_impossible': False}


In [13]:
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
import json
import random

# ── 1. Load annotation file ──────────────────────────────────────────────────
file_path = hf_hub_download(
    repo_id="theatticusproject/cuad",
    filename="CUAD_v1/CUAD_v1.json",
    repo_type="dataset"
)

with open(file_path) as f:
    cuad = json.load(f)

print(f"Contracts loaded: {len(cuad['data'])}")

# ── 2. Target clause categories ──────────────────────────────────────────────
TARGET_CATEGORIES = [
    "Governing Law",
    "Termination For Convenience",
    "Liquidated Damages",
    "Non-Compete",
    "Limitation Of Liability",
    "Arbitration",
]

def get_category(question: str) -> str | None:
    """Extract category name from CUAD question string."""
    q = question.lower()
    for cat in TARGET_CATEGORIES:
        if cat.lower() in q:
            return cat
    return None

# ── 3. Convert SQuAD-style → instruction format ──────────────────────────────
def build_instruction(category: str, clause: str, answer_spans: list[str]) -> dict:
    has_clause = len(answer_spans) > 0
    label = "Yes" if has_clause else "No"

    if has_clause:
        response = (
            f"Yes. The clause contains a '{category}' provision. "
            f"Relevant excerpt: \"{answer_spans[0][:300]}\""
        )
    else:
        response = f"No. The clause does not contain a '{category}' provision."

    return {
        "instruction": (
            f"Review the following contract clause and identify whether it "
            f"contains a '{category}' provision."
        ),
        "input": clause[:1200],   # truncate to avoid token overflow
        "output": response,
        "label": label,           # kept for eval metrics
        "category": category,
    }

# ── 4. Parse all contracts ────────────────────────────────────────────────────
all_samples = []

for contract in cuad["data"]:
    for para in contract["paragraphs"]:
        context = para["context"]
        for qa in para["qas"]:
            cat = get_category(qa["question"])
            if cat is None:
                continue

            answers = qa["answers"]
            spans = [a["text"] for a in answers]
            sample = build_instruction(cat, context, spans)
            all_samples.append(sample)

print(f"Total samples: {len(all_samples)}")

# ── 5. Class balance check ────────────────────────────────────────────────────
yes_count = sum(1 for s in all_samples if s["label"] == "Yes")
no_count  = len(all_samples) - yes_count
print(f"Yes: {yes_count} ({yes_count/len(all_samples):.1%})  |  No: {no_count} ({no_count/len(all_samples):.1%})")

# ── 6. Downsample 'No' to balance classes (2:1 max) ──────────────────────────
yes_samples = [s for s in all_samples if s["label"] == "Yes"]
no_samples  = [s for s in all_samples if s["label"] == "No"]

random.seed(42)
no_samples_downsampled = random.sample(no_samples, min(len(no_samples), len(yes_samples) * 2))
balanced = yes_samples + no_samples_downsampled
random.shuffle(balanced)

print(f"Balanced dataset size: {len(balanced)}")

# ── 7. Train / val / test split (80 / 10 / 10) ───────────────────────────────
train_val, test = train_test_split(balanced, test_size=0.10, random_state=42)
train, val      = train_test_split(train_val, test_size=0.111, random_state=42)  # 0.111 * 0.9 ≈ 0.10

print(f"Train: {len(train)}  |  Val: {len(val)}  |  Test: {len(test)}")

# ── 8. Save splits ────────────────────────────────────────────────────────────
for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
    with open(f"{split_name}.json", "w") as f:
        json.dump(split_data, f, indent=2)
    print(f"Saved {split_name}.json")

# ── 9. Sanity check: print one example ───────────────────────────────────────
ex = train[0]
print("\n── Sample ──────────────────────────────")
print(f"### Instruction:\n{ex['instruction']}")
print(f"\n### Contract Clause:\n{ex['input'][:300]}...")
print(f"\n### Response:\n{ex['output']}")
print(f"\nCategory: {ex['category']}  |  Label: {ex['label']}")

Contracts loaded: 510
Total samples: 2550
Yes: 876 (34.4%)  |  No: 1674 (65.6%)
Balanced dataset size: 2550
Train: 2040  |  Val: 255  |  Test: 255
Saved train.json
Saved val.json
Saved test.json

── Sample ──────────────────────────────
### Instruction:
Review the following contract clause and identify whether it contains a 'Termination For Convenience' provision.

### Contract Clause:
Exhibit 99.4 JOINT FILING AGREEMENT In accordance with Rule 13d-1(k)(1) promulgated under the Securities Exchange Act of 1934, as amended, the undersigned hereby agree to the joint filing with each of the Reporting Persons (as such term is defined in the Schedule 13D referred to below) on behalf of ...

### Response:
No. The clause does not contain a 'Termination For Convenience' provision.

Category: Termination For Convenience  |  Label: No


In [16]:
# ── Step 2: W&B setup + zero-shot baseline ───────────────────────────────────
!pip install wandb -q

import wandb
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from sklearn.metrics import classification_report

# ── 2.1 Initialize W&B project ───────────────────────────────────────────────
wandb.init(
    project="cuad-lora-finetuning",
    name="baseline-zeroshot-Qwen/Qwen2.5-1.5B-Instruct",
    config={
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "dataset": "CUAD",
        "split": "test",
        "num_samples": 255,
        "experiment_type": "zero_shot_baseline",
    }
)

# ── 2.2 Load test set ─────────────────────────────────────────────────────────
with open("test.json") as f:
    test_data = json.load(f)

# ── 2.3 Load base model ───────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

# ── 2.4 Inference function ────────────────────────────────────────────────────
def predict(sample: dict, max_new_tokens: int = 100) -> str:
    prompt = (
        f"### Instruction:\n{sample['instruction']}\n\n"
        f"### Contract Clause:\n{sample['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()

def extract_label(response: str) -> str:
    """Extract Yes/No from model response."""
    r = response.strip().lower()
    if r.startswith("yes"):
        return "Yes"
    elif r.startswith("no"):
        return "No"
    # Fallback: search anywhere in response
    if "yes" in r[:50]:
        return "Yes"
    return "No"

# ── 2.5 Run baseline eval ─────────────────────────────────────────────────────
y_true, y_pred = [], []
results_table = wandb.Table(columns=["category", "label", "prediction", "response", "correct"])

for i, sample in enumerate(test_data):
    response = predict(sample)
    pred = extract_label(response)
    correct = pred == sample["label"]

    y_true.append(sample["label"])
    y_pred.append(pred)
    results_table.add_data(
        sample["category"], sample["label"], pred, response[:200], correct
    )

    if i % 20 == 0:
        print(f"[{i+1}/{len(test_data)}] Label: {sample['label']} | Pred: {pred}")

# ── 2.6 Compute + log metrics ─────────────────────────────────────────────────
report = classification_report(y_true, y_pred, output_dict=True)

wandb.log({
    "accuracy":      report["accuracy"],
    "f1_yes":        report["Yes"]["f1-score"],
    "f1_no":         report["No"]["f1-score"],
    "f1_macro":      report["macro avg"]["f1-score"],
    "precision_yes": report["Yes"]["precision"],
    "recall_yes":    report["Yes"]["recall"],
    "predictions_table": results_table,
})

print("\n── Baseline Results ─────────────────────────────────────────────────")
print(classification_report(y_true, y_pred))

wandb.finish()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[1/255] Label: No | Pred: No
[21/255] Label: No | Pred: No
[41/255] Label: Yes | Pred: No
[61/255] Label: Yes | Pred: No
[81/255] Label: No | Pred: No
[101/255] Label: Yes | Pred: No
[121/255] Label: No | Pred: No
[141/255] Label: No | Pred: No
[161/255] Label: Yes | Pred: No
[181/255] Label: No | Pred: No
[201/255] Label: No | Pred: No
[221/255] Label: No | Pred: Yes
[241/255] Label: No | Pred: No

── Baseline Results ─────────────────────────────────────────────────
              precision    recall  f1-score   support

          No       0.69      0.72      0.71       169
         Yes       0.41      0.37      0.39        86

    accuracy                           0.60       255
   macro avg       0.55      0.55      0.55       255
weighted avg       0.60      0.60      0.60       255



accuracy,▁
f1_macro,▁
f1_no,▁
f1_yes,▁
precision_yes,▁
recall_yes,▁
accuracy,0.60392
f1_macro,0.54756
f1_no,0.70725
f1_yes,0.38788
precision_yes,0.40506


## Fine-tuning Experiment

In [48]:
# ── Full Experiment 2: All-in-one ─────────────────────────────────────────────
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
from sklearn.metrics import classification_report
import torch, json, random, wandb

RANK = 16
LR   = 5e-5

# 1. Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=1792,
    dtype=torch.float16,
    load_in_4bit=True,
)

# 2. Attach LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=RANK * 2,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 3. Load balanced training data
PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Contract Clause:
{input}

### Response:
{output}"""

def format_sample(sample):
    return {"text": PROMPT_TEMPLATE.format(**sample)}

with open("train_balanced.json") as f:
    train_data = json.load(f)
with open("val.json") as f:
    val_data = json.load(f)

train_ds = Dataset.from_list(train_data).map(format_sample)
val_ds   = Dataset.from_list(val_data).map(format_sample)

print(f"Train size: {len(train_ds)}  Val size: {len(val_ds)}")

# 4. Train
wandb.init(
    project="cuad-lora-finetuning",
    name=f"lora-rank{RANK}-lr{LR}-balanced",
    config={
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "rank": RANK, "lora_alpha": RANK * 2,
        "learning_rate": LR, "epochs": 3,
        "train_strategy": "oversampled_yes_1to1",
    }
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=1792,
    args=TrainingArguments(
    output_dir=f"./outputs/rank{RANK}-lr{LR}-balanced",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="steps",        # changed from evaluation_strategy
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    seed=42,

    ),
)

trainer.train()
print("Last 3 log entries:", trainer.state.log_history[-3:])
wandb.finish()

# 5. Eval
FastLanguageModel.for_inference(model)

def predict(sample, max_new_tokens=100):
    prompt = (
        f"### Instruction:\n{sample['instruction']}\n\n"
        f"### Contract Clause:\n{sample['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extract_label(response):
    r = response.strip().lower()
    if r.startswith("yes"): return "Yes"
    if r.startswith("no"):  return "No"
    if "yes" in r[:50]:     return "Yes"
    return "No"

with open("test.json") as f:
    test_data = json.load(f)

y_true, y_pred = [], []
for i, sample in enumerate(test_data):
    pred = extract_label(predict(sample))
    y_true.append(sample["label"])
    y_pred.append(pred)
    if i % 20 == 0:
        print(f"[{i+1}/{len(test_data)}] Label: {sample['label']} | Pred: {pred}")

report = classification_report(y_true, y_pred, output_dict=True)
print(classification_report(y_true, y_pred))

wandb.init(project="cuad-lora-finetuning", name=f"eval-rank{RANK}-lr{LR}-balanced")
wandb.log({
    "accuracy": report["accuracy"],
    "f1_yes":   report["Yes"]["f1-score"],
    "f1_no":    report["No"]["f1-score"],
    "f1_macro": report["macro avg"]["f1-score"],
})
wandb.finish()

==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/2662 [00:00<?, ? examples/s]

Map:   0%|          | 0/255 [00:00<?, ? examples/s]

...

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

[241/255] Label: No | Pred: No


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

              precision    recall  f1-score   support

          No       0.86      0.79      0.82       169
         Yes       0.64      0.74      0.69        86

    accuracy                           0.77       255
   macro avg       0.75      0.77      0.75       255
weighted avg       0.78      0.77      0.78       255



accuracy,▁
f1_macro,▁
f1_no,▁
f1_yes,▁
accuracy,0.77255
f1_macro,0.75458
f1_no,0.82099
f1_yes,0.68817


In [49]:
# ── Error Analysis ────────────────────────────────────────────────────────────
import json, torch

with open("test.json") as f:
    test_data = json.load(f)

# Collect all predictions with full details
results = []
for sample in test_data:
    response = predict(sample)
    pred = extract_label(response)
    results.append({
        "category":  sample["category"],
        "label":     sample["label"],
        "pred":      pred,
        "correct":   pred == sample["label"],
        "input":     sample["input"],
        "response":  response,
    })

# Separate error types
false_negatives = [r for r in results if r["label"] == "Yes" and r["pred"] == "No"]
false_positives = [r for r in results if r["label"] == "No"  and r["pred"] == "Yes"]

print(f"False Negatives (missed clause):   {len(false_negatives)}")
print(f"False Positives (hallucinated):    {len(false_positives)}")

# Error breakdown by category
print("\n── Errors by Category ───────────────────────────────────────────────")
from collections import Counter
fn_cats = Counter(r["category"] for r in false_negatives)
fp_cats = Counter(r["category"] for r in false_positives)
for cat in sorted(set(list(fn_cats) + list(fp_cats))):
    print(f"  {cat:<35} FN={fn_cats.get(cat,0)}  FP={fp_cats.get(cat,0)}")

# Print 3 most interesting failure cases
print("\n── Failure Case 1: False Negative ──────────────────────────────────")
fn = false_negatives[0]
print(f"Category : {fn['category']}")
print(f"Label    : {fn['label']}  →  Pred: {fn['pred']}")
print(f"Clause   : {fn['input'][:500]}")
print(f"Response : {fn['response']}")

print("\n── Failure Case 2: False Negative (different category) ─────────────")
# Pick one from a different category than case 1
fn2 = next(r for r in false_negatives if r["category"] != fn["category"])
print(f"Category : {fn2['category']}")
print(f"Label    : {fn2['label']}  →  Pred: {fn2['pred']}")
print(f"Clause   : {fn2['input'][:500]}")
print(f"Response : {fn2['response']}")

print("\n── Failure Case 3: False Positive ───────────────────────────────────")
fp = false_positives[0]
print(f"Category : {fp['category']}")
print(f"Label    : {fp['label']}  →  Pred: {fp['pred']}")
print(f"Clause   : {fp['input'][:500]}")
print(f"Response : {fp['response']}")

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

False Negatives (missed clause):   22
False Positives (hallucinated):    36

── Errors by Category ───────────────────────────────────────────────
  Governing Law                       FN=0  FP=9
  Liquidated Damages                  FN=5  FP=1
  Non-Compete                         FN=11  FP=8
  Termination For Convenience         FN=6  FP=18

── Failure Case 1: False Negative ──────────────────────────────────
Category : Termination For Convenience
Label    : Yes  →  Pred: No
Clause   : Exhibit 10.21

Confidential treatment has been requested for portions of this exhibit. The copy filed herewith omits the information subject to the confidentiality request. Omissions are designated as [*****]. A complete version of this exhibit has been filed separately with the Securities and Exchange Commission.

Network Management Outsourcing Agreement



Bank of South Pacific Ltd   Network Management Outsourcing Agreement







Datec Contact Details   [*****]   [*****]   [*****]   [*****]  
Respon